# FinSight AI — Notebook 05: Temporal Validation & Distribution Stability

Validates that data partitioning strictly respects temporal causality, verifies zero lookahead leakage, and tracks distribution drift with PSI and ADWIN.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))

import pandas as pd
from finsight.ml_core.drift_detection import calculate_psi, ADWINDriftDetector

tx_path = Path.cwd().parent / 'data' / 'synthetic' / 'transactions.csv'
df = pd.read_csv(tx_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
print(f'Date range: {df["timestamp"].min()} to {df["timestamp"].max()}')


## 1. Population Stability Index (PSI) Across Temporal Halves


In [ ]:
midpoint = len(df) // 2
ref_spend = df.iloc[:midpoint]['amount'].values
curr_spend = df.iloc[midpoint:]['amount'].values

psi = calculate_psi(ref_spend, curr_spend)
print(f'Population Stability Index (Amount): {psi:.4f}')
if psi < 0.1:
    print('Status: STABLE (No significant distribution drift detected)')
elif psi < 0.2:
    print('Status: MODERATE SHIFT (Monitor feature inputs)')
else:
    print('Status: SIGNIFICANT DRIFT (Model retraining advised)')
